# Follow-up Adequacy
**We examine censoring rates by diagnosis year to assess whether patients have sufficient follow-up to produce reliable pseudo-observations at 2 years.**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
df = pd.read_csv('../outputs/ioio_tki_features_df.csv', dtype = dtype_map)

In [3]:
df.shape

(1339, 174)

## Censoring rates by years

In [4]:
df.groupby('adv_diagnosis_year')['event'].apply(lambda x: (x == 0).mean())

adv_diagnosis_year
2011    0.470588
2012    0.473684
2013    0.261905
2014    0.230769
2015    0.321168
2016    0.372263
2017    0.390977
2018    0.439716
2019    0.446970
2020    0.500000
2021    0.519084
2022    0.696970
2023    0.861111
Name: event, dtype: float64

In [5]:
(df.query('adv_diagnosis_year >= 2022')['event'] == 0).mean()

np.float64(0.7549019607843137)

In [6]:
results = []
for year in sorted(df['adv_diagnosis_year'].unique()):
    censored = df.query('adv_diagnosis_year == @year and event == 0', engine='python')
    if len(censored) > 0:
        frac = (censored['duration'] < 730).mean()
        results.append({'year': year, 'frac_censored_before_2y': frac})

pd.DataFrame(results)

,year,frac_censored_before_2y
0,2011,0.250000
1,2012,0.333333
2,2013,0.272727
3,2014,0.333333
4,2015,0.363636
5,2016,0.254902
6,2017,0.288462
7,2018,0.306452
8,2019,0.271186
9,2020,0.366197


**Among patients diagnosed with advanced disease in 2022 and later, >70% are censored and all lack 2-year follow-up from the data cut, making 2-year RMST pseudo-observations entirely extrapolated for these patients. The primary analysis is therefore restricted to patients diagnosed in 2021 or earlier, where the majority of patients have reached the 2-year endpoint and pseudo-observations are grounded in observed rather than extrapolated survival.**